In [ ]:
# Sprint 2 – Simple model and initial evaluation
# Deliverable 2 – Building an initial neural network:
# 1. Choice of network architecture
# ● Objective: Identify the deep learning architecture best suited to the nature of the data and the prediction task.
# ● Architectures considered:
# MLP: for tabular data without sequential or spatial dependencies.
# CNN / ResNet / DenseNet: for spatially structured data (images, signals).
# RNN / LSTM / GRU: if the data has temporal dependencies.
# Transformers: for long sequences or complex data (text, time series).
# GNN: for relational data (graphs).
# ● Final decision:
# Initial implementation with an MLP (Multi-Layer Perceptron), adapted to tabular data.
# ● Justification:
# Simple architecture to establish a robust baseline.
# Ease of interpretation of layers.
# 2. Construction of the neural network (MLP)
# ● Implemented structure:
# Input: input_shape = 10 (10 explanatory variables)
# Hidden layer 1: 8 neurons – ReLU activation
# Hidden layer 2: 4 neurons – ReLU activation
# Output layer: 1 neuron – tanh activation (for binary or centered continuous prediction)
# ● Technical choices:
# Loss function: binary_crossentropy or MSE depending on the nature of the target
# Optimizer: Adam
# Weight initialization: He normal
# ● Justification:
# ReLU activation promotes non-linearity and limits gradient issues.
# Architecture is sufficiently lightweight to avoid initial overfitting.
 
# 3. Training and evaluation
# ● Training phase:
# Epochs = 50, batch_size = 32 (as an example)
# Early stopping on validation loss
# ● Performance evaluation:
# AUC-ROC: for the model's ability to distinguish between classes
# Confusion matrix: to assess accuracy, recall, and specificity
# ● Justification:
# AUC is more robust in the context of unbalanced classes
# 4. Analysis of the decision threshold
# ● Objective: Optimize the classification threshold in relation to a business objective (precision, F1, recall, etc.)
# ● Method:
# Generation of the F1 curve based on the decision threshold
# Choice of optimal threshold = maximum F1 or according to business constraints (min FP, max recall, etc.)
# ● Justification:
# Promotes a compromise tailored to the criticality of false positives/negatives.
# 5. Monitoring of experiments
# ● Implementation of minimum traceability of experiments
# Automatic logs (via TensorBoard, MLFlow, or simple CSV/logs)
# Model versioning: backup in .h5 or .pt format with timestamp
# Backup of training parameters (architecture, seed, hyperparams)
# ● Comparative experiments (at least 3 architectures):
# MLP baseline (simple, shallow)
# Extended MLP (deeper layers or dropout)
# ResNet-like Dense MLP (with skip connections on dense layers)

# ● Justification:
# Assess robustness, generalizability, and computational costs
# 6. Interim report
# ● Contents:
# Presentation of performance metrics (AUC, F1, precision, recall, etc.)
# Justified technical choices for each version
# Comparative analysis of results
# Proposed areas for improvement:
# ■   	Hyperparameter optimization (GridSearch, Optuna, etc.)
# ■   	Testing advanced architectures (Transformers, TabNet, etc.)
# ■   	Feature engineering or enhanced encoding

In [3]:
import pandas as pd
# We will use the processed data from the first deliverable noted processed_X.csv and processed_y.csv
# Load the data
X = pd.read_csv('processed_X.csv')
y = pd.read_csv('processed_y.csv')


In [2]:
# print the shapes of the data
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (236182, 21)
Shape of y: (236182, 1)


In [4]:
# split the data into training and testing sets
from sklearn.model_selection import train_test_split 
X_temp, X_val, y_temp, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)


In [5]:
%pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [12]:
# Building the MLP model 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense 
import keras
model = Sequential([
    Dense(8, activation='relu', input_shape=(X_train.shape[1], )),
    Dense(4, activation='relu'),
    Dense(1, activation='tanh')
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['AUC'])

model.summary()

/home/ahmedou/.local/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 8)              │           176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 4)              │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 217 (868.00 B)

 Trainable params: 217 (868.00 B)

 Non-trainable params: 0 (0.00 B)

In [14]:
# Train the model 
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=50, batch_size=32, verbose=0)

# Evaluate the model on the test set using AUC-ROC 
test_loss, test_auc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test AUC: {test_auc:.4f}')

Test AUC: 0.8178
